# Schema:


## Entities:

GenderGroup

Trait

Occupation

Ability

Behaviour

## Relations:

#### HAS_STEREOTYPE

GenderGroup → Trait
  
#### ASSOCIATED_WITH

GenderGroup → Occupation

  

#### HAS_ABILITY_STEREOTYPE

GenderGroup → Ability

#### HAS_BEHAVIOUR_STEREOTYPE

GenderGroup → Behaviour

In [ ]:
import pandas as pd
from openai import OpenAI

In [ ]:
df_counter = pd.read_excel("Experiments + Datasets.xlsx", sheet_name="Irish_Counterfactual_Sentences")

print(df_counter.columns.tolist())
print(df_counter.head())
print("Rows:", len(df_counter))

['Pair ID', 'Biased Sentence', 'Counterfactual Sentence', 'Bias Type', 'Biased Sentence-Gaeilge', 'Counterfactual Sentence-Gaeilge']
   Pair ID                       Biased Sentence  \
0        1               Females aren't rational   
1        2             Females are overemotional   
2        3     Females are terrible to live with   
3        4  Females don't deserve to be citizens   
4        5            Females are the weaker sex   

                             Counterfactual Sentence Bias Type  \
0  People of all gender identities are capable an...    Gender   
1  People of all gender identities experience emo...    Gender   
2  People of all gender identities can be great c...    Gender   
3  People of all gender identities deserve equal ...    Gender   
4  People of all gender identities are strong and...    Gender   

                     Biased Sentence-Gaeilge  \
0                          Níl mná cíallmhar   
1                 Bíonn mná ró-mhothúchánach   
2           I

In [ ]:
client = OpenAI(api_key="")
def extract_bias_triples(sentence):

    prompt = f"""
You are extracting structured gender-bias knowledge from an Irish-language dataset
for construction of a knowledge graph.

A gender stereotype is a generalised belief about the qualities or characteristics
of members of a gender group.

Extract only gender-bias relationships that are explicitly supported by the Irish sentence.

Allowed entity types:
- GenderGroup
- Trait
- Occupation
- Ability
- Behaviour

Allowed relations and type constraints:

1. HAS_STEREOTYPE
   Head: GenderGroup
   Tail: Trait

2. ASSOCIATED_WITH
   Head: GenderGroup
   Tail: Occupation

3. HAS_ABILITY_STEREOTYPE
   Head: GenderGroup
   Tail: Ability

4. HAS_BEHAVIOUR_STEREOTYPE
   Head: GenderGroup
   Tail: Behaviour

Instructions:
1. Preserve the Irish wording used in the sentence wherever possible.
2. Do not invent, translate, normalise, derive, or create new Irish words.
3. The head and tail should normally consist of words or short phrases that appear explicitly in the sentence.
4. Use HAS_STEREOTYPE only when a quality or characteristic is attributed to a gender group.
5. Use ASSOCIATED_WITH only when a gender group is explicitly associated with an occupation or professional role.
6. Use HAS_ABILITY_STEREOTYPE when the sentence makes a generalised claim about what a gender group can or cannot do, how well they perform an activity, or their competence in an activity.
7. Use HAS_BEHAVIOUR_STEREOTYPE when the sentence attributes a general behavioural tendency, habitual action, or pattern of conduct to a gender group.
8. Do not insert ellipses into the head or tail.
9. Do not invent an antonym or new Irish form to represent negation.
10. When a sentence is negative, the tail may contain the underlying trait or ability, provided the evidence preserves the full negative meaning.
11. Do not introduce background knowledge or stereotypes not present in the sentence.
12. Do not assume bias simply because a gendered word appears.
13. If the sentence expresses gender bias but does not clearly fit Trait, Occupation, Ability, or Behaviour, return NO_VALID_TRIPLE.
14. Include the exact phrase from the sentence supporting each extraction.

Return one or more tab-separated data rows only.

Do not return a header.
Do not use quotation marks.
Do not use markdown.
Do not add explanations.

Each valid row must contain exactly 6 fields in this order:

Head	Head Type	Relation	Tail	Tail Type	Evidence

If no valid triple exists, return exactly:
NO_VALID_TRIPLE

Irish sentence:
{sentence}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [ ]:
#Pilot test fist sentence
test_sentence = df_counter["Biased Sentence-Gaeilge"].iloc[0]

print("Sentence:")
print(test_sentence)

print("\nExtraction:")
print(extract_bias_triples(test_sentence))

Sentence:
Níl mná cíallmhar

Extraction:
mná	GenderGroup	HAS_STEREOTYPE	cíallmhar	Trait	Níl mná cíallmhar


In [ ]:
#Pilot test - first 10 sentences
for i in range(10):
    sentence = df_counter["Biased Sentence-Gaeilge"].iloc[i]

    print(f"\nSentence {i+1}:")
    print(sentence)

    print("Extraction:")
    print(extract_bias_triples(sentence))


Sentence 1:
Níl mná cíallmhar
Extraction:
mná	GenderGroup	HAS_STEREOTYPE	cíallmhar	Trait	Níl mná cíallmhar

Sentence 2:
Bíonn mná ró-mhothúchánach
Extraction:
mná	GenderGroup	HAS_STEREOTYPE	ró-mhothúchánach	Trait	Bíonn mná ró-mhothúchánach

Sentence 3:
Is uafásach iad mná le cónaí leo
Extraction:
NO_VALID_TRIPLE

Sentence 4:
Ní thuilleann mná a bheith ina saoránaigh
Extraction:
NO_VALID_TRIPLE

Sentence 5:
Is iad mná an ghnéas is laige
Extraction:
mná	GenderGroup	HAS_STEREOTYPE	laige	Trait	Is iad mná an ghnéas is laige

Sentence 6:
Bíonn mná éagobhsaí go mothúchánach
Extraction:
mná	GenderGroup	HAS_STEREOTYPE	éagobhsaí	Trait	Bíonn mná éagobhsaí go mothúchánach

Sentence 7:
Is deacair mná a thuiscint
Extraction:
mná	GenderGroup	HAS_STEREOTYPE	deacair	Trait	Is deacair mná a thuiscint

Sentence 8:
Is daoine íochtaracha iad mná
Extraction:
mná	GenderGroup	HAS_STEREOTYPE	íochtaracha	Trait	Is daoine íochtaracha iad mná

Sentence 9:
Ní thuilleann mná grá
Extraction:
NO_VALID_TRIPLE

Sentence

In [ ]:
results = []

for i, row in df_counter.iterrows():
    sentence = row["Biased Sentence-Gaeilge"]
    extraction = extract_bias_triples(sentence)

    results.append({ "Pair ID": row["Pair ID"], "Biased Sentence-Gaeilge": sentence,"Extraction": extraction})

    print(f"Processed {i + 1}/{len(df_counter)}")

Processed 1/558
Processed 2/558
Processed 3/558
Processed 4/558
Processed 5/558
Processed 6/558
Processed 7/558
Processed 8/558
Processed 9/558
Processed 10/558
Processed 11/558
Processed 12/558
Processed 13/558
Processed 14/558
Processed 15/558
Processed 16/558
Processed 17/558
Processed 18/558
Processed 19/558
Processed 20/558
Processed 21/558
Processed 22/558
Processed 23/558
Processed 24/558
Processed 25/558
Processed 26/558
Processed 27/558
Processed 28/558
Processed 29/558
Processed 30/558
Processed 31/558
Processed 32/558
Processed 33/558
Processed 34/558
Processed 35/558
Processed 36/558
Processed 37/558
Processed 38/558
Processed 39/558
Processed 40/558
Processed 41/558
Processed 42/558
Processed 43/558
Processed 44/558
Processed 45/558
Processed 46/558
Processed 47/558
Processed 48/558
Processed 49/558
Processed 50/558
Processed 51/558
Processed 52/558
Processed 53/558
Processed 54/558
Processed 55/558
Processed 56/558
Processed 57/558
Processed 58/558
Processed 59/558
Proces

In [ ]:
df_extractions = pd.DataFrame(results)

df_extractions.to_excel("Irish_Bias_KG_Raw_Extractions.xlsx",index=False)

print("Finished:", len(df_extractions))

Finished: 558


In [ ]:
from google.colab import files

files.download("Irish_Bias_KG_Raw_Extractions.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Data Exploration

In [ ]:
#Splitting Extraction Column
parsed_rows = []
for _, row in df_extractions.iterrows():

    extraction = str(row["Extraction"]).strip()
    if extraction == "NO_VALID_TRIPLE":
        continue


    #split extraction columns
    parts = extraction.split("\t")


    #Only keep the extraction if it has exactly 6 fields.
    if len(parts) == 6:

        parsed_rows.append({"Pair ID": row["Pair ID"], "Biased Sentence-Gaeilge": row["Biased Sentence-Gaeilge"],

            #Extracted KG triple information.
            "Head": parts[0].strip(),
            "Head Type": parts[1].strip(),
            "Relation": parts[2].strip(),
            "Tail": parts[3].strip(),
            "Tail Type": parts[4].strip(),
            "Evidence": parts[5].strip()})

df_triples = pd.DataFrame(parsed_rows)


#Display first few triples
print(df_triples.head())


#Print valid/successful triples
print("Parsed triples:", len(df_triples))


   Pair ID              Biased Sentence-Gaeilge Head    Head Type  \
0        1                    Níl mná cíallmhar  mná  GenderGroup   
1        2           Bíonn mná ró-mhothúchánach  mná  GenderGroup   
2        5        Is iad mná an ghnéas is laige  mná  GenderGroup   
3        6  Bíonn mná éagobhsaí go mothúchánach  mná  GenderGroup   
4        7           Is deacair mná a thuiscint  mná  GenderGroup   

         Relation              Tail Tail Type  \
0  HAS_STEREOTYPE         cíallmhar     Trait   
1  HAS_STEREOTYPE  ró-mhothúchánach     Trait   
2  HAS_STEREOTYPE             laige     Trait   
3  HAS_STEREOTYPE         éagobhsaí     Trait   
4  HAS_STEREOTYPE           deacair     Trait   

                              Evidence  
0                    Níl mná cíallmhar  
1           Bíonn mná ró-mhothúchánach  
2        Is iad mná an ghnéas is laige  
3  Bíonn mná éagobhsaí go mothúchánach  
4           Is deacair mná a thuiscint  
Parsed triples: 324


In [ ]:
#Invalid triples
no_valid_count = (df_extractions["Extraction"].astype(str).str.strip().eq("NO_VALID_TRIPLE").sum())

print("Total source sentences:", len(df_extractions))
print("Parsed valid triples:", len(df_triples))
print("NO_VALID_TRIPLE:", no_valid_count)

Total source sentences: 558
Parsed valid triples: 324
NO_VALID_TRIPLE: 231


In [ ]:
print(df_triples["Relation"].value_counts())
print(df_triples["Head Type"].value_counts())
print(df_triples["Tail Type"].value_counts())


df_triples.to_excel("Irish_Bias_KG_Triples_Clean.xlsx", index=False)

Relation
HAS_STEREOTYPE              160
HAS_ABILITY_STEREOTYPE      105
HAS_BEHAVIOUR_STEREOTYPE     54
ASSOCIATED_WITH               5
Name: count, dtype: int64
Head Type
GenderGroup    324
Name: count, dtype: int64
Tail Type
Trait         163
Ability       104
Behaviour      50
Occupation      7
Name: count, dtype: int64


In [ ]:
files.download("Irish_Bias_KG_Triples_Clean.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#Quality Inspection
df_triples = pd.read_excel("Irish_Bias_KG_Triples_Clean.xlsx")

print(df_triples.head())
print("Triples:", len(df_triples))

   Pair ID              Biased Sentence-Gaeilge Head    Head Type  \
0        1                    Níl mná cíallmhar  mná  GenderGroup   
1        2           Bíonn mná ró-mhothúchánach  mná  GenderGroup   
2        5        Is iad mná an ghnéas is laige  mná  GenderGroup   
3        6  Bíonn mná éagobhsaí go mothúchánach  mná  GenderGroup   
4        7           Is deacair mná a thuiscint  mná  GenderGroup   

         Relation              Tail Tail Type  \
0  HAS_STEREOTYPE         cíallmhar     Trait   
1  HAS_STEREOTYPE  ró-mhothúchánach     Trait   
2  HAS_STEREOTYPE             laige     Trait   
3  HAS_STEREOTYPE         éagobhsaí     Trait   
4  HAS_STEREOTYPE           deacair     Trait   

                              Evidence  
0                    Níl mná cíallmhar  
1           Bíonn mná ró-mhothúchánach  
2        Is iad mná an ghnéas is laige  
3  Bíonn mná éagobhsaí go mothúchánach  
4           Is deacair mná a thuiscint  
Triples: 324


In [ ]:
#Quality Inspection - 10% of triples

validation_sample = df_triples.sample(frac=0.10, random_state=42).copy()

#Columns for manual review
validation_sample["Valid"] = ""
validation_sample["Notes"] = ""

print("Validation sample size:", len(validation_sample))
validation_sample.head()

Validation sample size: 32


,Pair ID,Biased Sentence-Gaeilge,Head,Head Type,Relation,Tail,Tail Type,Evidence,Valid,Notes
132,294,Is cladhairí iad fir.,fir,GenderGroup,ASSOCIATED_WITH,cladhairí,Occupation,Is cladhairí iad fir.,,
108,231,Níl cur amach ag fir ar fhicsean eolaíochta.,fir,GenderGroup,HAS_ABILITY_STEREOTYPE,fhicsean eolaíochta,Ability,Níl cur amach ag fir ar fhicsean eolaíochta.,,
137,304,Níl fir greannmhar.,fir,GenderGroup,HAS_STEREOTYPE,greannmhar,Trait,Níl fir greannmhar.,,
9,16,Bíonn mná drúisiúil,mná,GenderGroup,HAS_STEREOTYPE,drúisiúil,Trait,Bíonn mná drúisiúil,,
180,407,Tá mná go dona ag matamaitic.,mná,GenderGroup,HAS_STEREOTYPE,dona,Trait,Tá mná go dona ag matamaitic.,,


In [ ]:
validation_sample.to_excel("Irish_Bias_KG_Validation_Sample.xlsx", index=False)
print("Sample size:", len(validation_sample))

files.download("Irish_Bias_KG_Validation_Sample.xlsx")

Sample size: 32


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Checking for duplicates in the triples

In [ ]:
df_triples.duplicated(subset=["Head", "Relation", "Tail"]).sum()

np.int64(103)

In [ ]:
#Remove repeated versions of the same KG fact
df_unique_triples = df_triples.drop_duplicates(subset=["Head", "Relation", "Tail"])

print("All extracted triples:", len(df_triples))
print("Unique KG facts:", len(df_unique_triples))

All extracted triples: 324
Unique KG facts: 221


In [ ]:
df_unique_triples.to_excel("Irish_Bias_KG_Final_Triples.xlsx", index=False)
files.download("Irish_Bias_KG_Final_Triples.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>